# Day 2 - Topic 3: Closures and Decorators

> Lead-Level Data Science Interview Prep Series

## 1. Introduction

- A **closure** is an inner function that remembers variables from its enclosing (outer) function, even after the outer function has finished running
- A **decorator** is a function that takes another function, adds extra behavior around it, and returns the enhanced version - built on top of closures
- Why needed?
  - Closures: create functions with "remembered settings" without classes or globals
  - Decorators: add cross-cutting behavior (timing, logging, caching, validation) to many functions WITHOUT touching their code
- Where used?
  - `@lru_cache` for caching expensive computations
  - `@app.route(...)` in Flask/FastAPI, `@pytest.fixture` in testing, `@staticmethod`/`@property` in classes
  - Timing/logging decorators in real ML pipelines - a Lead-level code-quality signal

## 2. Real-Life Analogy

- **Closure** = a backpack: when the inner function leaves the outer function's "house", it carries a backpack containing the outer variables it needs. The house is demolished (outer function ends), but the backpack stays with the function forever
- **Decorator** = a phone case: the phone (original function) still works exactly the same, but the case ADDS protection/features around it without modifying the phone internally. You can put the same style of case on many different phones
- Gift-wrapping is the same idea: the gift is unchanged, the wrapper adds something around it - and `@decorator` is just the wrapping station

## 3. Explanation

- **Closure requirements (all 3):**
  1. A nested (inner) function
  2. The inner function references variables of the outer function
  3. The outer function RETURNS the inner function
- The returned inner function keeps access to those outer variables - they are stored in `__closure__`
- **Decorator flow:**
  1. Decorator receives a function as input
  2. Defines a `wrapper` function that runs extra code before/after calling the original
  3. Returns `wrapper`
- `@decorator_name` above a `def` is pure syntax sugar for `func = decorator_name(func)`
- `*args, **kwargs` in the wrapper let it decorate ANY function regardless of its signature

> **Trick to remember:** Decorator = closure + a function as the "remembered" variable. If you understand the backpack, the phone case follows.

## 4. Syntax

```python
# Closure
def outer(setting):
    def inner(x):
        return x * setting     # inner remembers 'setting'
    return inner               # return the function itself (no parentheses)

# Decorator
import functools

def my_decorator(func):
    @functools.wraps(func)                 # preserves func's name/docstring
    def wrapper(*args, **kwargs):
        # code BEFORE original runs
        result = func(*args, **kwargs)     # call the original
        # code AFTER original runs
        return result
    return wrapper

@my_decorator
def greet(name):
    return f"Hello, {name}"

# @my_decorator is identical to writing: greet = my_decorator(greet)
```

- `functools.wraps(func)` - copies the original function's metadata (`__name__`, docstring) onto the wrapper - always include it

In [ ]:
# Minimal closure
def make_multiplier(factor):
    def multiply(x):
        return x * factor
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(10), triple(10))    # 20 30 - each closure carries its own 'factor'
print(double.__closure__[0].cell_contents)   # peek inside the backpack: 2


## 5. Examples

### Basic Example

In [ ]:
# Basic: simplest decorator - announce before and after
def announce(func):
    def wrapper():
        print("Before the call")
        func()
        print("After the call")
    return wrapper

@announce
def say_hi():
    print("Hi!")

say_hi()


### Intermediate Example

In [ ]:
# Intermediate: a timing decorator that works on ANY function
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))


- `*args, **kwargs` make the wrapper universal - it forwards whatever arguments the original expects
- `functools.wraps` keeps `slow_sum.__name__` as "slow_sum" instead of "wrapper" - critical for debugging
- Timing how long each pipeline step takes is a genuinely useful real-world decorator you can reuse across every project

### Real-World Example

In [ ]:
# Real-world: caching expensive computation with the built-in lru_cache decorator
import functools
import time

@functools.lru_cache(maxsize=None)
def expensive_score(user_id):
    time.sleep(1)                  # simulate slow computation / DB call
    return user_id * 42

start = time.time()
print(expensive_score(7))          # slow - actually computes
print(f"First call: {time.time() - start:.2f}s")

start = time.time()
print(expensive_score(7))          # instant - served from cache
print(f"Second call: {time.time() - start:.4f}s")


- `@lru_cache` memoizes results: same input -> cached output, skipping recomputation entirely
- LRU = Least Recently Used - when full (`maxsize`), it evicts the least recently used entries first
- Classic uses: recursive Fibonacci (exponential -> linear), repeated feature lookups, any pure function that is called repeatedly with the same inputs
- Caveat: arguments must be hashable (no lists/dicts as arguments), and it trades memory for speed

## 6. Internal Working

- Normally, a function's local variables die when the function returns
- With a closure, Python detects that the inner function references outer variables ("free variables") and stores them in special **cell objects**, attached to the inner function's `__closure__` attribute
- To REASSIGN (not just read) an outer variable inside the inner function, you need the `nonlocal` keyword - otherwise Python creates a new local variable instead
- When decorating: `@decorator` runs AT DEFINITION TIME (import time), replacing the name with the wrapper - the wrapper then runs at every call

> **Trick to remember:** Reading outer variables is free; reassigning them needs `nonlocal`. Decorators wrap once (at definition), run every call.

In [ ]:
# nonlocal - closure that keeps mutable state
def make_counter():
    count = 0
    def counter():
        nonlocal count      # without this line: UnboundLocalError
        count += 1
        return count
    return counter

c = make_counter()
print(c(), c(), c())    # 1 2 3 - state persists between calls, no class needed


## 7. Time and Space Complexity

- Closure variable access: O(1) - a direct cell lookup
- Decorator overhead per call: O(1) - one extra function call layer
- `lru_cache`: O(1) average per lookup/insert (hash table underneath); space O(k) where k = number of cached entries - the classic space-for-time trade
- Decorators do not change the decorated function's own complexity - they add a constant wrapper cost

## 8. Common Mistakes

- Returning `inner()` (calling it) instead of `inner` (the function object) from the outer function
- Forgetting `@functools.wraps(func)` - then `__name__`/docstring show "wrapper", breaking debugging and documentation
- Forgetting to `return result` inside the wrapper - the decorated function silently starts returning `None`
- Trying to reassign an outer variable without `nonlocal` - `UnboundLocalError`
- Using `@lru_cache` on functions with list/dict arguments (unhashable -> TypeError) or on impure functions whose results change between calls

In [ ]:
# Mistake: wrapper that forgets to return
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)      # result thrown away!
    return wrapper

@bad_decorator
def add(a, b):
    return a + b

print(add(2, 3))    # None - the classic silent bug


## 9. Best Practices

- Always use `@functools.wraps(func)` in every decorator you write
- Always `return func(*args, **kwargs)`'s result from the wrapper
- Keep decorators single-purpose (one for timing, one for logging) - stack them if needed; note stacking order: the decorator closest to `def` is applied first
- Use `@lru_cache` only on pure functions (same input always gives same output) with hashable arguments
- Use closures for small stateful behavior; if state/methods grow, switch to a class

## 10. Interview Questions

**Beginner**
- Q: What is a closure?
  A: An inner function that remembers and can access variables from its enclosing function's scope even after the outer function has finished executing.
- Q: What does the @ symbol do?
  A: It applies a decorator - `@dec` above `def f` is shorthand for `f = dec(f)`.

**Intermediate**
- Q: Why do decorator wrappers use `*args, **kwargs`?
  A: So one decorator can wrap any function regardless of its parameter signature, forwarding all positional and keyword arguments to the original unchanged.
- Q: What is the purpose of functools.wraps?
  A: It copies the original function's metadata (name, docstring, etc.) onto the wrapper, so the decorated function still identifies as itself in debugging, help(), and tracebacks.

**Advanced**
- Q: What is the nonlocal keyword and when is it required?
  A: It declares that a variable inside a nested function refers to the enclosing function's variable, allowing reassignment. Without it, assignment creates a new local variable (or raises UnboundLocalError if read-then-assigned).
- Q: How does @lru_cache change the complexity of recursive Fibonacci?
  A: Naive recursion recomputes subproblems, giving O(2^n) time. With lru_cache each distinct input is computed once and then served from an O(1) hash lookup, reducing time to O(n) at the cost of O(n) cache space.

## 11. Practice Problems

**Easy**
1. Write a closure `make_greeter(greeting)` that returns a function greeting any name with that fixed greeting.
2. Write a decorator that prints the name of the function every time it is called (use functools.wraps).

**Medium**
3. Write a `make_counter()` closure supporting increment and reporting the current count using nonlocal.
4. Write a decorator `log_calls` that prints the arguments a function was called with and the value it returned.

**Hard**
5. Write a `retry` decorator that re-runs the decorated function up to 3 times if it raises an exception, printing each attempt number - then explain in a comment why `*args, **kwargs` and returning the result are both essential here.

## 12. Revision Summary

- Closure = inner function + outer variables remembered ("backpack"), returned from outer
- Read outer vars freely; reassigning them needs `nonlocal`
- Decorator = function that wraps a function and returns the wrapper; `@dec` means `f = dec(f)`
- Wrapper template: `*args, **kwargs` in, `return func(*args, **kwargs)` out, `@functools.wraps` on top
- Decoration happens once at definition; the wrapper runs at every call
- `@lru_cache` = built-in memoization decorator - space-for-time trade, hashable args only
- Forgetting wraps or the return statement are the two classic decorator bugs

> **Next topic (Day 2 continues):** Classes and Objects